# 06 · Modelo extra · Naive Bayes gaussiano

El enunciado pide probar **algo que no hayamos visto en clase** y demostrar si mejora o no. En clase hemos dado regresión, KNN, SVM, métodos de ensemble (Random Forest, XGBoost) y redes neuronales (MLP). Aquí probamos una familia distinta a todas ellas: **Naive Bayes**, un clasificador *probabilístico generativo*.

## Qué es y por qué lo probamos

Los modelos anteriores buscan directamente una frontera entre "contrata" y "no contrata". Naive Bayes hace otra cosa: aprende **cómo son los clientes de cada clase** (la media y la varianza de cada variable entre los que contratan y entre los que no) y, para un cliente nuevo, aplica el teorema de Bayes:

$$P(\text{contrata} \mid x) \propto P(\text{contrata}) \cdot \prod_i P(x_i \mid \text{contrata})$$

Se llama *naive* (ingenuo) porque supone que las variables son **independientes entre sí** dentro de cada clase, y por eso puede multiplicar las probabilidades de cada una por separado. `GaussianNB` supone además que cada variable sigue una distribución normal.

Motivos para probarlo en este problema:

- **Devuelve una probabilidad `p`**, que es lo que necesitan las reglas de negocio.
- **Es rapidísimo y casi no tiene hiperparámetros**: no hay árboles, ni épocas, ni distancias que calcular. Si empatara con Random Forest sería el modelo más simple.
- **Suele dar buen recall en clases minoritarias**, que es la métrica que prioriza el banco.
- **Sabemos de antemano dónde puede fallar**: en el EDA vimos que `euribor3m`, `nr.employed` y `emp.var.rate` están correlacionadas entre 0,91 y 0,97. Eso rompe la suposición de independencia, así que el experimento sirve también para comprobar cuánto le afecta.

Se evalúa igual que el resto: mismo preprocesado, validación cruzada estratificada de 5 pliegues solo sobre **train**, con y sin SMOTE, y todo registrado en MLflow.

## 0. Librerías

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.naive_bayes import GaussianNB

# Raíz del proyecto (el notebook se ejecuta desde notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.utils import (
    cargar_datos_procesados,
    crear_preprocesador,
    evaluar_cv,
    registrar_en_mlflow,
)

SEMILLA = 42

## 1. Datos

Los mismos datos procesados que usa el resto del equipo (`data/processed`). Solo se usa **train**: el test se descarta y no se evalúa en este notebook.

In [2]:
X_train, _, y_train, _ = cargar_datos_procesados()  # el test no se toca

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"Proporción positiva en train: {y_train.mean():.3%}")

X_train: (32940, 21) | y_train: (32940,)
Proporción positiva en train: 11.266%


## 2. Pipeline

Reutilizamos el preprocesado común de `src/utils.py` (`StandardScaler` + `OneHotEncoder`). Dos detalles propios de este modelo:

- **`sparse_threshold=0`**: `GaussianNB` no acepta matrices dispersas y el one-hot puede devolver una. Con este parámetro el preprocesador devuelve siempre una matriz densa; las transformaciones son exactamente las mismas.
- **SMOTE dentro del pipeline** (`imblearn.pipeline.Pipeline`), en el orden *preprocesado → SMOTE → modelo*, para que los clientes sintéticos se generen solo con el train de cada pliegue y la validación se haga con datos reales (sin fuga de datos).

En Naive Bayes, SMOTE actúa sobre todo cambiando la **probabilidad a priori**: sin SMOTE el modelo parte de que solo el 11 % contrata; con SMOTE parte de un 50 %, así que se atreve más a decir "contrata".

In [3]:
def crear_pipeline_nb(var_smoothing: float, usar_smote: bool) -> ImbPipeline:
    """Pipeline preprocesado común -> (SMOTE) -> GaussianNB, sin entrenar.

    Args:
        var_smoothing: fracción de la mayor varianza que se suma a todas las varianzas.
        usar_smote: si True, sobremuestrea la clase minoritaria dentro de cada pliegue.
    """
    pasos = [("preprocesado", crear_preprocesador().set_params(sparse_threshold=0))]
    if usar_smote:
        pasos.append(("smote", SMOTE(random_state=SEMILLA)))
    pasos.append(("modelo", GaussianNB(var_smoothing=var_smoothing)))
    return ImbPipeline(pasos)

## 3. Experimentos: `var_smoothing`, con y sin SMOTE

`GaussianNB` solo tiene un hiperparámetro relevante, **`var_smoothing`**: una cantidad que se suma a la varianza de todas las variables para estabilizar el cálculo.

- **Valor muy pequeño** (`1e-9`, por defecto): el modelo se fía de la varianza que ha medido. Las columnas one-hot poco frecuentes (por ejemplo `poutcome = success`) tienen una varianza diminuta y disparan la probabilidad en cuanto aparecen.
- **Valor grande** (`1.0`): todas las campanas de Gauss se ensanchan, las variables discriminan menos y el modelo se vuelve más conservador.

Probamos 6 valores en escala logarítmica, cada uno con y sin SMOTE: **12 ejecuciones**, evaluadas con `evaluar_cv` (la misma validación cruzada de 5 pliegues y las mismas métricas que el resto del proyecto) y registradas en MLflow.

In [4]:
VALORES_VAR_SMOOTHING = [1e-9, 1e-5, 1e-3, 1e-2, 1e-1, 1.0]

resultados = []
for usar_smote in (False, True):
    for vs in VALORES_VAR_SMOOTHING:
        metricas = evaluar_cv(crear_pipeline_nb(vs, usar_smote), X_train, y_train, semilla=SEMILLA)
        resultados.append({"smote": usar_smote, "var_smoothing": vs, **metricas})

df_resultados = pd.DataFrame(resultados)
columnas_metricas = [c for c in df_resultados.columns if c not in ("smote", "var_smoothing")]

# El mejor candidato de cada condición (por F1 de validación, como en el resto del
# proyecto) se registra con un nombre limpio; el resto, como candidatos.
mejores = df_resultados.groupby("smote")["f1_val"].idxmax()
for indice, fila in df_resultados.iterrows():
    base = f"naive_bayes_{'smote' if fila['smote'] else 'sin_smote'}"
    nombre = base if indice in mejores.values else f"{base}_candidato_vs_{fila['var_smoothing']:g}"
    registrar_en_mlflow(
        nombre,
        params={
            "modelo": "GaussianNB", "var_smoothing": fila["var_smoothing"],
            "smote": bool(fila["smote"]), "cv_folds": 5, "random_state": SEMILLA,
        },
        metricas={c: float(fila[c]) for c in columnas_metricas},
    )

formato = {c: "{:.3f}" for c in columnas_metricas}
df_resultados.sort_values("f1_val", ascending=False).reset_index(drop=True).style.format(
    {**formato, "var_smoothing": "{:g}"}
)

,smote,var_smoothing,recall_train,recall_val,precision_train,precision_val,f1_train,f1_val,tiempo_entrenamiento_s
0,False,0.001,0.448,0.448,0.405,0.405,0.426,0.425,0.220
1,False,1e-05,0.455,0.454,0.397,0.396,0.424,0.423,0.379
2,False,0.01,0.463,0.459,0.392,0.390,0.425,0.422,0.156
3,True,0.001,0.535,0.534,0.349,0.348,0.422,0.421,0.283
4,True,1e-05,0.533,0.531,0.348,0.346,0.421,0.419,0.274
5,True,0.01,0.587,0.584,0.328,0.326,0.421,0.418,0.267
6,True,1,0.394,0.394,0.438,0.438,0.415,0.415,0.223
7,True,0.1,0.630,0.629,0.309,0.309,0.415,0.414,0.230
8,False,0.1,0.446,0.445,0.385,0.384,0.413,0.412,0.128
9,False,1e-09,0.582,0.577,0.321,0.319,0.413,0.411,0.279


### Lectura de los experimentos

- **No sobreajusta nada.** En las 12 ejecuciones las métricas de train y de validación coinciden hasta el segundo decimal. Es un modelo muy rígido (solo guarda una media y una varianza por variable y clase): su problema no es memorizar, es que **se queda corto**.
- **`var_smoothing` mueve el punto de equilibrio, no el techo.** Con el valor por defecto (`1e-9`) consigue más recall y menos precisión; con `1.0` se vuelve conservador (recall ≈ 0,26 y precisión ≈ 0,52 sin SMOTE). Pero el F1 apenas cambia: se queda entre 0,40 y 0,43 en casi todos los casos.
- **SMOTE sube el recall y baja la precisión**, igual que en el MLP y en XGBoost, y deja el F1 prácticamente igual. Con SMOTE y `var_smoothing = 0.1` llega al recall más alto del notebook (≈ 0,63).
- **Es el modelo más rápido del proyecto**: menos de un segundo por pliegue.

## 4. Comparación con lo que teníamos

Comparamos Naive Bayes con el modelo elegido del proyecto, Random Forest con `class_weight="balanced"` y sin SMOTE. De Naive Bayes mostramos la mejor configuración por F1 con y sin SMOTE, y además la de mayor recall, que es la métrica del negocio. Las métricas de Random Forest **no se recalculan aquí**: se leen de `models/mejor_modelo_info.json`, que genera `src/entrenar_modelo.py` con la misma validación cruzada.

In [5]:
info_rf = json.loads((RAIZ / "models" / "mejor_modelo_info.json").read_text(encoding="utf-8"))

filas = [{"modelo": "Random Forest (elegido)", "smote": False, **info_rf["metricas_cv"]}]
for indice in mejores.values:
    fila = df_resultados.loc[indice]
    filas.append({
        "modelo": f"Naive Bayes, mejor F1 (var_smoothing={fila['var_smoothing']:g})",
        **fila.drop("var_smoothing").to_dict(),
    })
fila = df_resultados.loc[df_resultados["recall_val"].idxmax()]
filas.append({
    "modelo": f"Naive Bayes, máximo recall (var_smoothing={fila['var_smoothing']:g})",
    **fila.drop("var_smoothing").to_dict(),
})

df_comparacion = pd.DataFrame(filas)
df_comparacion.style.format(formato)

,modelo,smote,recall_train,recall_val,precision_train,precision_val,f1_train,f1_val,tiempo_entrenamiento_s
0,Random Forest (elegido),False,0.713,0.610,0.451,0.401,0.553,0.483,8.996
1,"Naive Bayes, mejor F1 (var_smoothing=0.001)",False,0.448,0.448,0.405,0.405,0.426,0.425,0.220
2,"Naive Bayes, mejor F1 (var_smoothing=0.001)",True,0.535,0.534,0.349,0.348,0.422,0.421,0.283
3,"Naive Bayes, máximo recall (var_smoothing=0.1)",True,0.630,0.629,0.309,0.309,0.415,0.414,0.230


## 5. Conclusión: ¿mejora o no?

**No mejora. Mantenemos Random Forest como modelo elegido.**

- **Por F1**, la mejor configuración de Naive Bayes (sin SMOTE, `var_smoothing = 0.001`) se queda en **0,425** frente a **0,483** de Random Forest, con menos recall (0,448 frente a 0,610) y la misma precisión (0,405 frente a 0,401).
- **Por recall**, Naive Bayes con SMOTE y `var_smoothing = 0.1` llega a **0,629**, algo por encima de Random Forest (0,610) y similar al SVM lineal (0,622). Pero lo paga en precisión: **0,309** frente a 0,401. Para detectar prácticamente los mismos clientes, el banco tendría que hacer alrededor de un tercio más de llamadas. Con un recall casi igual, gana el modelo que gasta menos llamadas.
- **Su única ventaja es la velocidad** (menos de 1 s frente a varios segundos), que con 33.000 filas no es un problema que tengamos.

**Por qué se queda corto.** La suposición de independencia no se cumple en nuestros datos:

- Las variables económicas (`euribor3m`, `nr.employed`, `emp.var.rate`) dicen casi lo mismo (correlación 0,91-0,97). Naive Bayes las trata como tres pruebas independientes y **cuenta la misma evidencia tres veces**, así que sus probabilidades se van a los extremos. Lo mismo pasa con `pdays`, `previous`, `contactado_antes`, `contactos_totales` y `poutcome`.
- Las columnas one-hot valen 0 o 1 y no siguen una distribución normal, que es lo que supone `GaussianNB`.
- No puede aprender **interacciones** (por ejemplo, "estudiante *y* euríbor bajo"), que es justo lo que hacen bien los árboles de Random Forest.

Es un resultado útil aunque no gane: confirma que la ventaja de Random Forest no viene solo de compensar el desbalanceo, sino de combinar variables, y que en este dataset un modelo que trata cada variable por separado tiene el techo en un F1 de ≈ 0,43.

Como el modelo nuevo no supera al elegido, **no se toca** `models/mejor_modelo.joblib`, ni la app, ni las reglas.

## 6. MLflow

Las 12 ejecuciones quedan en el experimento común `reto1-deposito-banco`. Las dos principales se llaman `naive_bayes_sin_smote` y `naive_bayes_smote`; el resto llevan el sufijo `_candidato_vs_<valor>`. Para verlas, desde la raíz del proyecto: `mlflow ui --backend-store-uri sqlite:///mlflow.db`.